# ASR Model Evaluation — English

Three tiers, each answering a different question:

| Tier | Data | Answers |
|---|---|---|
| **A** | ~50 People's Speech clips (15-20s) | **Primary WER.** Enough samples for a stable number. |
| **B** | s1 + s5 (hand-verified) | **WER on two-party conversation** with a reference you trust. |
| **C** | 3 AMI files x 2 bandwidths (17-26 min) | **RTFx, VRAM, long-audio behaviour, telephony penalty.** |

Headline metrics: **WER** and **RTFx**. S/D/I and VRAM collected for the appendix.

Run cells top to bottom. Restart the session after Cell 1.

In [ ]:
# CELL 1 — install, then Runtime > Restart session
!pip install -q "nemo_toolkit[asr]" faster-whisper jiwer soundfile librosa \
    pandas matplotlib seaborn tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2

In [ ]:
# CELL 2 — mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -o -q "/content/drive/MyDrive/asr_benchmark/people_speech.zip" -d /content/ps
!ls /content/ps

people_speech


In [ ]:
# CELL 3 — CONFIG
from pathlib import Path

BASE_DIR   = Path("/content/drive/MyDrive/asr_benchmark")
SHORT_AUDIO_DIR = Path("/content/ps/people_speech/peoples_speech_audio")
SHORT_TEXT_DIR  = Path("/content/ps/people_speech/peoples_speech_transcriptions")
OUT_DIR    = BASE_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHORT_BATCH_SIZE = 8      # batch short clips - this is what real throughput looks like
LONG_CHUNK_S     = 480    # 8 min chunks for the long AMI files
LONG_OVERLAP_S   = 10

# Tier B - verified conversational references
TIER_B = [
    ("s1", "s1.wav", "s1.txt"),
    ("s5", "s5.wav", "s5.txt"),
]

# Tier C - long-form, speed + bandwidth
TIER_C = [
    ("ES2002a",    "ES2002a.Array1-01.wav",    "ES2002a_reference.txt", "16kHz"),
    ("ES2004a",    "ES2004a.Array1-01.wav",    "ES2004a_reference.txt", "16kHz"),
    ("IS1000a",    "IS1000a.Array1-01.wav",    "IS1000a_reference.txt", "16kHz"),
    ("ES2002a_8k", "ES2002a.Array1-01_8k.wav", "ES2002a_reference.txt", "8kHz"),
    ("ES2004a_8k", "ES2004a.Array1-01_8k.wav", "ES2004a_reference.txt", "8kHz"),
    ("IS1000a_8k", "IS1000a.Array1-01_8k.wav", "IS1000a_reference.txt", "8kHz"),
]

#MODELS_TO_RUN = ["parakeet", "canary", "faster-whisper"]
MODELS_TO_RUN = ["canary", "faster-whisper"]

In [ ]:
!ls /content/ps/people_speech/peoples_speech_transcriptions | head -3Z

In [ ]:
# CELL 4 — discover files and sanity-check before spending GPU time
import re

def find_transcript(audio_path):
    stem = audio_path.stem
    for cand in [SHORT_TEXT_DIR / f"{stem}.txt",
                 SHORT_TEXT_DIR / f"{stem}_flac.txt",
                 SHORT_TEXT_DIR / f"{audio_path.name}.txt"]:
        if cand.exists():
            return cand
    return None

def discover_short():
    if not SHORT_AUDIO_DIR.exists():
        print(f"  !! {SHORT_AUDIO_DIR} not found"); return []
    out, orphans = [], []
    for a in sorted(SHORT_AUDIO_DIR.glob("*.flac")) + sorted(SHORT_AUDIO_DIR.glob("*.wav")):
        t = find_transcript(a)
        if t: out.append((a.stem, a, t))
        else: orphans.append(a.name)
    if orphans:
        print(f"  !! {len(orphans)} orphans, e.g. {orphans[:2]}")
        print(f"     transcripts dir has: {[p.name for p in list(SHORT_TEXT_DIR.iterdir())[:2]]}")
    return out

SHORT_CLIPS = discover_short()
print(f"Tier A: {len(SHORT_CLIPS)} clip/transcript pairs")

for label, group in [("Tier B", TIER_B), ("Tier C", TIER_C)]:
    missing = [w for row in group for w in row[1:3] if not (BASE_DIR / w).exists()]
    print(f"{label}: {len(group) - len(missing)}/{len(group)*2} paths OK"
          + (f"  MISSING: {missing}" if missing else ""))

Tier A: 257 clip/transcript pairs
Tier B: 2/4 paths OK
Tier C: 6/12 paths OK


In [ ]:
# CELL 5 — audio + scoring helpers
import time, gc, json
import numpy as np, pandas as pd, soundfile as sf, librosa, torch, jiwer
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

normalizer = EnglishTextNormalizer({})
SR = 16000
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY")

def read_audio(path):
    a, sr = sf.read(str(path), dtype="float32")
    if a.ndim > 1:
        a = a.mean(axis=1)
    if sr != SR:
        a = librosa.resample(a, orig_sr=sr, target_sr=SR)
    return a

def chunk(audio, win_s=LONG_CHUNK_S, hop_s=None):
    hop_s = hop_s or (win_s - LONG_OVERLAP_S)
    w, s, pos = win_s * SR, hop_s * SR, 0
    while pos < len(audio):
        c = audio[pos:pos + w]
        if len(c) < SR:
            break
        yield c
        pos += s

def score(ref_text, hyp_text):
    ref = normalizer(ref_text)
    hyp = normalizer(hyp_text)
    m = jiwer.process_words(ref, hyp or "\u2205")
    return {"S": m.substitutions, "D": m.deletions, "I": m.insertions,
            "N": m.hits + m.substitutions + m.deletions,
            "hyp_words": len(hyp.split())}, hyp

GPU: Tesla T4


In [ ]:
# CELL 6 — model wrappers
class Parakeet:
    key, label = "parakeet", "Parakeet-TDT-0.6B-v2"
    def load(self):
        import nemo.collections.asr as na
        self.m = na.models.ASRModel.from_pretrained("nvidia/parakeet-tdt-0.6b-v2").eval().cuda()
        try:
            self.m.change_attention_model("rel_pos_local_attn", [128, 128])
            self.m.change_subsampling_conv_chunking_factor(1)
        except Exception as e:
            print("   long-audio mode unavailable:", e)
    def _t(self, arrays, bs):
        with torch.inference_mode():
            o = self.m.transcribe(arrays, batch_size=bs, verbose=False)
        return [x.text if hasattr(x, "text") else str(x) for x in o]
    def batch(self, arrays):  return self._t(arrays, SHORT_BATCH_SIZE)
    def long(self, audio):    return " ".join(self._t(list(chunk(audio)), 1))

class Canary(Parakeet):
    key, label = "canary", "Canary-1B-Flash"
    def load(self):
        import nemo.collections.asr as na
        self.m = na.models.ASRModel.from_pretrained("nvidia/canary-1b-flash").eval().cuda()
    def _t(self, arrays, bs):
        with torch.inference_mode():
            o = self.m.transcribe(
                arrays, batch_size=bs, verbose=False,
                source_lang="en", target_lang="en",
                task="asr", pnc="yes",
            )
        return [x.text if hasattr(x, "text") else str(x) for x in o]
    def long(self, audio):
        return " ".join(self._t(list(chunk(audio, 30, 28)), 4))

class FasterWhisper:
    key, label = "faster-whisper", "faster-whisper-large-v3"
    def load(self):
        from faster_whisper import WhisperModel
        self.m = WhisperModel("large-v3", device="cuda", compute_type="float16")
    def _one(self, a):
        segs, _ = self.m.transcribe(a, language="en", beam_size=5)
        return " ".join(s.text.strip() for s in segs)
    def batch(self, arrays):  return [self._one(a) for a in arrays]
    def long(self, audio):    return self._one(audio)   # handles long audio natively

REGISTRY = {c.key: c for c in [Parakeet, Canary, FasterWhisper]}

In [ ]:
# CELL 7 — RUN (30-60 min on a T4)


def measure(fn):
    """Run fn, return (result, wall_seconds, peak_vram_gb)."""
    if torch.cuda.is_available():
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    r = fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    el = time.perf_counter() - t0
    v = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.0
    return r, el, v

for key in MODELS_TO_RUN:
    mdl = REGISTRY[key]()
    print(f"\n{'='*60}\n{mdl.label}\n{'='*60}")
    try:
        mdl.load()
    except Exception as e:
        print("  LOAD FAILED:", e); continue

    # warmup - discarded. First call pays CUDA init + cuDNN autotune.
    try:
        mdl.batch([np.zeros(SR*5, dtype=np.float32)])
    except Exception as e:
        print("  warmup skipped:", e)

    # ---- TIER A : short clips, batched ----
    if SHORT_CLIPS:
        arrays = [read_audio(p) for _, p, _ in SHORT_CLIPS]
        total_audio = sum(len(a) for a in arrays) / SR
        hyps, el, vram = measure(lambda: mdl.batch(arrays))
        agg = {"S":0,"D":0,"I":0,"N":0}
        for (cid, _, tpath), h in zip(SHORT_CLIPS, hyps):
            s, hn = score(tpath.read_text(encoding="utf-8", errors="replace"), h)
            transcripts[f"{mdl.label}|A|{cid}"] = hn
            for k in agg: agg[k] += s[k]
        rows.append({"model": mdl.label, "tier": "A", "item": f"{len(arrays)} clips",
                     "band": "16kHz", "audio_s": round(total_audio,1), **agg,
                     "infer_s": round(el,2), "vram_gb": round(vram,2)})
        print(f"  TIER A  WER {100*(agg['S']+agg['D']+agg['I'])/agg['N']:.2f}%   "
              f"{total_audio/el:.1f}x realtime   {vram:.1f}GB")

    # ---- TIER B : verified conversational ----
    for cid, wav, ref in TIER_B:
        p = BASE_DIR / wav
        if not p.exists(): continue
        a = read_audio(p)
        h, el, vram = measure(lambda: mdl.long(a))
        s, hn = score((BASE_DIR/ref).read_text(encoding="utf-8", errors="replace"), h)
        transcripts[f"{mdl.label}|B|{cid}"] = hn
        rows.append({"model": mdl.label, "tier": "B", "item": cid, "band": "16kHz",
                     "audio_s": round(len(a)/SR,1), **{k:s[k] for k in ("S","D","I","N")},
                     "infer_s": round(el,2), "vram_gb": round(vram,2)})
        print(f"  TIER B  {cid:10s} WER {100*(s['S']+s['D']+s['I'])/s['N']:6.2f}%")

    # ---- TIER C : long-form, speed + bandwidth ----
    for cid, wav, ref, band in TIER_C:
        p = BASE_DIR / wav
        if not p.exists(): continue
        a = read_audio(p)
        h, el, vram = measure(lambda: mdl.long(a))
        s, hn = score((BASE_DIR/ref).read_text(encoding="utf-8", errors="replace"), h)
        transcripts[f"{mdl.label}|C|{cid}"] = hn
        dur = len(a)/SR
        rows.append({"model": mdl.label, "tier": "C", "item": cid, "band": band,
                     "audio_s": round(dur,1), **{k:s[k] for k in ("S","D","I","N")},
                     "infer_s": round(el,2), "vram_gb": round(vram,2)})
        print(f"  TIER C  {cid:12s} WER {100*(s['S']+s['D']+s['I'])/s['N']:6.2f}%   "
              f"{dur/el:6.1f}x   {vram:.1f}GB")

    del mdl; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df["wer"]      = (df.S + df.D + df.I) / df.N * 100
df["sub_rate"] = df.S / df.N * 100
df["del_rate"] = df.D / df.N * 100
df["ins_rate"] = df.I / df.N * 100
df["rtfx"]     = df.audio_s / df.infer_s
df = df.round(2)
df.to_csv(OUT_DIR/"per_item.csv", index=False)
(OUT_DIR/"transcripts.json").write_text(json.dumps(transcripts, indent=2))
print("\nDone.")


Canary-1B-Flash
[NeMo I 2026-09-01 11:49:57 mixins:218] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-09-01 11:49:57 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1152 tokens
[NeMo I 2026-09-01 11:49:57 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-09-01 11:49:57 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-09-01 11:49:57 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-09-01 11:49:57 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-09-01 11:49:57 aggregate_tokenizer:73] Aggregate vocab size: 5248


[NeMo W 2026-09-01 11:49:57 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 8
    pin_memory: true
    prompt_format: canary2
    max_tps: 25
    max_duration: 40.0
    text_field: answer
    lang_field: target_lang
    use_bucketing: true
    bucket_duration_bins:
    - - 3.971
      - 30
    - - 3.971
      - 48
    - - 4.973
      - 37
    - - 4.973
      - 60
    - - 5.85
      - 42
    - - 5.85
      - 71
    - - 6.56
      - 46
    - - 6.56
      - 79
    - - 7.32
      - 49
    - - 7.32
      - 88
    - - 8.19
      - 54
    - - 8.19
      - 99
    - - 8.88
      - 61
    - - 8.88
      - 107
    - - 9.76
      - 66
    - - 9.76
      - 118
    - - 10.56
      - 72
    -

[NeMo I 2026-09-01 11:50:22 save_restore_connector:287] Model EncDecMultiTaskModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--canary-1b-flash/snapshots/2b6e4d2dacb11cc1b1724de31bb48fe68c26c12e/canary-1b-flash.nemo.


[NeMo W 2026-09-01 11:50:22 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-09-01 11:50:26 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1


  TIER A  WER 7.12%   42.1x realtime   7.4GB


[NeMo W 2026-09-01 11:52:08 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1


... (11 outputs left out)

[NeMo W 2026-09-01 11:55:21 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1


  TIER C  ES2004a_8k   WER  40.17%     24.4x   4.1GB


[NeMo W 2026-09-01 11:56:06 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1


  TIER C  IS1000a_8k   WER  52.75%     29.6x   4.1GB

faster-whisper-large-v3
  TIER A  WER 7.28%   10.0x realtime   0.0GB
  TIER B  s1         WER  31.05%
  TIER B  s5         WER  23.18%
  TIER C  ES2002a      WER  36.43%      4.9x   0.0GB
  TIER C  ES2004a      WER  35.87%      5.0x   0.0GB
  TIER C  IS1000a      WER  56.86%      4.9x   0.0GB
  TIER C  ES2002a_8k   WER  44.43%      5.6x   0.0GB
  TIER C  ES2004a_8k   WER  35.24%      5.3x   0.0GB
  TIER C  IS1000a_8k   WER  53.52%      4.5x   0.0GB

Done.


In [ ]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/content/drive/MyDrive/asr_benchmark/results")
df = pd.read_csv(OUT_DIR / "per_item.csv")
print(len(df), "rows")   # expect 27

27 rows


In [ ]:
print(df.columns.tolist())

['model', 'tier', 'item', 'band', 'audio_s', 'S', 'D', 'I', 'N', 'infer_s', 'vram_gb', 'wer', 'sub_rate', 'del_rate', 'ins_rate', 'rtfx']


In [ ]:
# CELL 8 — aggregate tables
def rollup(sub):
    return pd.Series({
        "wer":      (sub.S.sum()+sub.D.sum()+sub.I.sum())/sub.N.sum()*100,
        "sub_rate": sub.S.sum()/sub.N.sum()*100,
        "del_rate": sub.D.sum()/sub.N.sum()*100,
        "ins_rate": sub.I.sum()/sub.N.sum()*100,
        "rtfx":     sub.audio_s.sum()/sub.infer_s.sum(),
        "vram_gb":  sub.vram_gb.max(),
        "ref_words": sub.N.sum(),
    })

by_tier = df.groupby(["model","tier"]).apply(rollup).round(2).reset_index()
by_band = df[df.tier=="C"].groupby(["model","band"]).apply(rollup).round(2).reset_index()

# HEADLINE: accuracy from Tier A (most samples), speed from Tier C (long files)
acc  = df[df.tier.isin(["A","B"])].groupby("model").apply(rollup).round(2)
spd  = df[(df.tier=="C") & (df.band=="16kHz")].groupby("model").apply(rollup).round(2)
headline = pd.DataFrame({"WER %": acc.wer, "RTFx": spd.rtfx,
                         "Peak VRAM GB": df.groupby("model").vram_gb.max()}).reset_index()

for name, t in [("BY TIER", by_tier), ("TIER C BY BANDWIDTH", by_band),
                ("HEADLINE", headline)]:
    print(f"\n{name}\n" + "-"*70 + "\n" + t.to_string(index=False))

by_tier.to_csv(OUT_DIR/"summary_by_tier.csv", index=False)
by_band.to_csv(OUT_DIR/"summary_by_band.csv", index=False)
headline.to_csv(OUT_DIR/"summary_headline.csv", index=False)

print("\nTELEPHONY PENALTY (Tier C, 8kHz minus 16kHz):")
for m in by_band.model.unique():
    s = by_band[by_band.model==m].set_index("band")
    if {"8kHz","16kHz"} <= set(s.index):
        print(f"  {m}: {s.loc['16kHz','wer']:.2f}% -> {s.loc['8kHz','wer']:.2f}%  "
              f"(+{s.loc['8kHz','wer']-s.loc['16kHz','wer']:.2f} pts)")

/tmp/ipykernel_1452/2948663944.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_tier = df.groupby(["model","tier"]).apply(rollup).round(2).reset_index()
/tmp/ipykernel_1452/2948663944.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_band = df[df.tier=="C"].groupby(["model","band"]).apply(rollup).round(2).reset_index()
/tmp/ipykernel_1452/2948663944.py:17: DeprecationWarning: DataFrameGroupBy.a


BY TIER
----------------------------------------------------------------------
                  model tier   wer  sub_rate  del_rate  ins_rate   rtfx  vram_gb  ref_words
        Canary-1B-Flash    A  7.12      3.13      2.34      1.65  42.09     7.39    10494.0
        Canary-1B-Flash    B 18.60      3.10      6.70      8.80  25.37     4.09     1000.0
        Canary-1B-Flash    C 46.31     10.47     28.33      7.50  29.59     4.09    15972.0
   Parakeet-TDT-0.6B-v2    A  7.83      3.10      2.71      2.03 171.77     5.43    10494.0
   Parakeet-TDT-0.6B-v2    B 14.30      4.20      7.40      2.70 129.53     4.46     1000.0
   Parakeet-TDT-0.6B-v2    C 35.52      9.74     20.62      5.17 154.89     5.45    15972.0
faster-whisper-large-v3    A  7.28      3.19      2.16      1.92  10.04     0.01    10494.0
faster-whisper-large-v3    B 24.90      5.40     15.50      4.00  11.98     0.01     1000.0
faster-whisper-large-v3    C 43.83     13.28     17.10     13.45   4.96     0.01    15972.0


In [ ]:
VRAM_OVERRIDE = {"faster-whisper-large-v3": 4.70}

In [ ]:
    ax.annotate(f'{r["model"]}\n{r["VRAM"]:.1f} GB',
                (r["RTFx"], r["WER %"]), xytext=(0,-34),
                textcoords="offset points", ha="center", fontsize=10)
ax.set_title("Accuracy vs. Speed\nbubble area proportional to peak VRAM")

NameError: name 'ax' is not defined

In [ ]:
# CELL 9 — figures (300 DPI)
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="talk")
pal = sns.color_palette("colorblind")

# Fig 1 - accuracy vs speed, bubble = VRAM
# Fig 1 - accuracy vs speed, bubble = VRAM
VRAM_OVERRIDE = {"faster-whisper-large-v3": 4.70}

acc_a = df[df.tier == "A"].groupby("model").apply(rollup).round(2)
spd_c = df[(df.tier == "C") & (df.band == "16kHz")].groupby("model").apply(rollup).round(2)
vram  = df.groupby("model").vram_gb.max()
plot_df = pd.DataFrame({"WER %": acc_a.wer, "RTFx": spd_c.rtfx, "VRAM": vram}).reset_index()
plot_df["VRAM"] = plot_df.apply(lambda r: VRAM_OVERRIDE.get(r["model"], r["VRAM"]), axis=1)

fig, ax = plt.subplots(figsize=(11,7))
for i,(_,r) in enumerate(plot_df.iterrows()):
    ax.scatter(r["RTFx"], r["WER %"], s=r["VRAM"]*250, alpha=.7,
               edgecolors="black", linewidth=1.4, zorder=3, color=pal[i])
    ax.annotate(f'{r["model"]}\n{r["VRAM"]:.1f} GB',
                (r["RTFx"], r["WER %"]), xytext=(0,-34),
                textcoords="offset points", ha="center", fontsize=10)
ax.set_xscale("log"); ax.margins(.3)
ax.set_xlabel("RTFx on 20-min audio, T4, batch=1 (higher = faster)")
ax.set_ylabel("WER % on Tier A, 257 clips (lower = better)")
ax.set_title("Accuracy vs. Speed\nbubble area proportional to peak VRAM")
fig.tight_layout(); fig.savefig(OUT_DIR/"fig1_accuracy_vs_speed.png", dpi=300)

# Fig 2 - error composition per model
piv = by_tier[by_tier.tier=="A"].set_index("model")[["sub_rate","del_rate","ins_rate"]]
fig, ax = plt.subplots(figsize=(10,6.5))
piv.plot(kind="bar", stacked=True, ax=ax, color=pal[:3],
         edgecolor="black", linewidth=.6)
ax.set_ylabel("% of reference words"); ax.set_xlabel("")
ax.set_title("Error Composition"); ax.legend(["Substitutions","Deletions","Insertions"])
plt.xticks(rotation=15, ha="left")
fig.tight_layout(); fig.savefig(OUT_DIR/"fig2_error_composition.png", dpi=300)

# Fig 3 - WER by tier (shows how much the AMI reference inflates things)
fig, ax = plt.subplots(figsize=(11,6.5))
sns.barplot(data=by_tier, x="tier", y="wer", hue="model", ax=ax,
            edgecolor="black", linewidth=.6, palette="colorblind")
for c in ax.containers: ax.bar_label(c, fmt="%.1f", fontsize=9, padding=2)
ax.set_ylabel("WER (%)"); ax.set_xlabel("Tier")
ax.set_title("WER by Data Tier\nA = short clips | B = verified conversation | C = AMI long-form")
fig.tight_layout(); fig.savefig(OUT_DIR/"fig3_wer_by_tier.png", dpi=300)

# Fig 4 - bandwidth effect
if not by_band.empty:
    fig, ax = plt.subplots(figsize=(10,6.5))
    sns.barplot(data=by_band, x="model", y="wer", hue="band", ax=ax,
                edgecolor="black", linewidth=.6, palette="colorblind")
    for c in ax.containers: ax.bar_label(c, fmt="%.1f", fontsize=9, padding=2)
    ax.set_ylabel("WER (%)"); ax.set_xlabel("")
    ax.set_title("Narrowband (8 kHz) Penalty")
    plt.xticks(rotation=15, ha="right")
    ax.legend(loc="lower right", title="band")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig4_bandwidth.png", dpi=300)

print("4 figures written to", OUT_DIR)

/tmp/ipykernel_1452/2207896221.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  acc_a = df[df.tier == "A"].groupby("model").apply(rollup).round(2)
/tmp/ipykernel_1452/2207896221.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spd_c = df[(df.tier == "C") & (df.band == "16kHz")].groupby("model").apply(rollup).round(2)


4 figures written to /content/drive/MyDrive/asr_benchmark/results


In [ ]:
import json, random
ts = json.loads((OUT_DIR/"transcripts.json").read_text())
for k in random.sample(list(ts), min(4, len(ts))):
    print(f"\n--- {k} ---\n{ts[k][:400]}")


--- faster-whisper-large-v3|A|07282016HFUUforum_SLASH_07-28-2016_HFUUforum_DOT_mp3_00004 ---
commons here in the spirit of being able to grow our food here locally so we started as an organization in actually in 2009 but really in 2010 march of 2010 we had our 1st meeting

--- faster-whisper-large-v3|A|07282016HFUUforum_SLASH_07-28-2016_HFUUforum_DOT_mp3_00096 ---
methods and methods that are better for us work then then they are more apt to use them and i would want to create policy to make us use them thank you thank you all very much we have one last

--- faster-whisper-large-v3|A|07282016HFUUforum_SLASH_07-28-2016_HFUUforum_DOT_mp3_00239 ---
allowed for both of you you will have one minute what will you do to initiate a citizens task force and request proposals for this for the state of the art treatment plants and

--- Canary-1B-Flash|A|07282016HFUUforum_SLASH_07-28-2016_HFUUforum_DOT_mp3_00087 ---
suspect to look at what the chemicals are in the area with that i have to say that